## Camada 2 - aprendizado few-shot (in-context learning)

Aqui está um dos achados centrais da era dos LLMs: o modelo aprende
dentro da própria conversa. Alguns exemplos rotulados no prompt
(*few-shot*) e a tarefa muda de comportamento - sem treinar nada.

A mudança em relação ao notebook anterior é pequena de propósito:
consertar um prompt fraco não é trocar de modelo, é dar o que faltou
- informação e exemplos que definam a regra.


In [ ]:
# Setup encapsulado: cada notebook é auto-contido (a ordem ENTRE
# notebooks é livre; dentro dele, as células rodam de cima para baixo)
import pathlib, sys
sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd
from llm import chat

df = pd.read_csv('../data/reclamacoes.csv')
CATEGORIAS = 'entrega | cobranca | produto | atendimento | app'

def prompt_zero_shot(texto: str) -> str:
    return (
        'Classifique a reclamacao em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [ ]:
# Os quatro exemplos abaixo são âncoras de regra escolhidas à mão:
# cada um ensina 'olhe além da palavra-chave: a causa raiz decide'.
# Escolher bons exemplos é o novo feature engineering: qualidade > quantidade.
FEW_SHOT = [
    ('O atendente disse que o reembolso do pedido que nao chegou foi aprovado e o boleto segue vindo', 'entrega'),
    ('A segunda parcela do pedido ficou mais cara que o combinado no carrinho', 'cobranca'),
    ('A camera parou de funcionar uma semana apos a compra', 'produto'),
    ('O atendente desligou o telefone na cara da cliente depois de 40 minutos de fila', 'atendimento'),
]

def prompt_few_shot(texto: str) -> str:
    exemplos = '\n'.join(f'Reclamacao: "{t}" -> {c}' for t, c in FEW_SHOT)
    return (
        f'{exemplos}\n\n'
        'Classifique pela CAUSA RAIZ da reclamacao, em UMA das categorias:\n'
        f'{CATEGORIAS}\n\n'
        f'Reclamacao: "{texto}"\n\n'
        'Responda apenas com a categoria, em minusculas, e nada mais.'
    )


In [ ]:
# Intervenções de palco:
# 1. rodar com os 4 exemplos e comparar com a camada 1
# 2. comentar uma âncora (ex.: a primeira, da linha 15) e ver a linha
#    correspondente regredir
INDICES = [15, 17]  # as armadilhas multimotivo; 15 e 17 têm âncora no few-shot

for i in INDICES:
    linha = df.iloc[i]
    pred = chat([{'role': 'user', 'content': prompt_few_shot(linha['texto'])}], temperature=0).strip().lower()
    ok = 'OK' if pred == linha['categoria'] else 'ERROU'
    print(f'[{ok}] {linha["texto"][:55]}...')
    print(f'      esperado: {linha["categoria"]} | veio: {pred}\n')


## Fechamento da camada 2

4 exemplos em 4 linhas mudaram o placar: as armadilhas que o
zero-shot errava agora seguem a regra. É o *in-context
learning* em ação: a 'inteligência' está no prompt, não nos
pesos do modelo.

E a mensagem de carreira: dados de qualidade vencem modelo
maior quase sempre - e custam muito menos.


## Bônus da camada 2 - micro-tuning: mexeu nos parâmetros e melhorou?

Intuição comum: 'se não ficou bom, mexe no temperature/top_p'.
A célula abaixo mede isso: 4 configurações, 3 rodadas, as linhas
15/16/17 (15 e 17 com âncora; 16 de propósito sem regra).

Expectativa: temp=0 repete a saída a cada rodada; temp>0 varia;
e o erro sistemático da 16 segue igual em qualquer config.

A leitura: parâmetros de amostragem compram determinismo, não
conhecimento. Falta de regra se conserta com exemplos no prompt.

(top_k só existe na API nativa do Gemini; não aparece neste
endpoint OpenAI-compat.)


In [ ]:
# A matriz do micro-tuning: 4 configs x 3 rodadas sobre as armadilhas.
# As rodadas repetidas existem para expor a VARIAÇÃO: com temp > 0,
# a mesma entrada pode voltar com respostas diferentes.
# temperature controla o acaso; top_p corta a cauda de improbáveis;
# top_k (não exposto neste endpoint) limitaria as k palavras mais prováveis
CONFIGS = [
    ('temp=0        ', dict(temperature=0)),
    ('temp=0.7      ', dict(temperature=0.7)),
    ('temp=0.7 p=0.3', dict(temperature=0.7, top_p=0.3)),
    ('temp=1.0 p=0.3', dict(temperature=1.0, top_p=0.3)),
]
RODADAS = 3

for nome_cfg, cfg in CONFIGS:
    acertos = 0
    for _ in range(RODADAS):
        for i in [15, 16, 17]:
            linha = df.iloc[i]
            pred = chat([{'role': 'user', 'content': prompt_few_shot(linha['texto'])}], **cfg).strip().lower()
            acertos += pred == linha['categoria']
    print(f'{nome_cfg} | placar: {acertos}/{RODADAS * 3}')

print()
print('Leitura: o placar empata entre configs; o que muda de verdade')
print('é a VARIAÇÃO das respostas entre rodadas quando temp > 0. E a')
print('linha 16 continua errando do mesmo jeito: o conserto dela exige')
print('exemplos que ensinem a regra - não parâmetros de amostragem.')
